# PanVision DenseNet embedding export

Run this after DenseNet training. It uses the frozen `best.pt` checkpoint and predicted-pancreas crops to create a training-only KNN reference bank plus separate validation and test query banks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPO_ROOT = Path('/content/Panvision')  # change if the repository is elsewhere
DATA_ROOT = Path('/content/drive/MyDrive/PanVisionData_Colab')
CROPS_DIR = DATA_ROOT / 'outputs/classification/predicted_crops'
RUN_DIR = DATA_ROOT / 'outputs/classification/training_runs/densenet121_predicted_RUN'
CHECKPOINT = RUN_DIR / 'best.pt'
SPLITS = REPO_ROOT / 'data/splits.csv'

for required in (REPO_ROOT / 'src/classification', CROPS_DIR, CHECKPOINT, SPLITS):
    assert required.exists(), f'Not found: {required}'
print('Checkpoint:', CHECKPOINT)
print('Crops:', CROPS_DIR)

In [ ]:
%pip install -q -r {REPO_ROOT / 'src/classification/requirements.txt'}

import sys
sys.path.insert(0, str(REPO_ROOT))
from src.classification.export_embeddings import export_dense_embeddings

metadata = export_dense_embeddings(
    checkpoint=CHECKPOINT,
    crops=CROPS_DIR,
    output=RUN_DIR,
    splits=SPLITS,
    batch_size=32,
    workers=2,
    device='auto',
    overwrite=False,
)
metadata

In [ ]:
import json
import numpy as np

with np.load(RUN_DIR / 'patient_embeddings_train.npz', allow_pickle=False) as bank:
    assert bank['embeddings'].ndim == 2 and bank['embeddings'].shape[1] == 512
    assert set(bank['split'].tolist()) == {'train'}
    assert bool(bank['knn_reference'])
    print('KNN reference shape:', bank['embeddings'].shape)

print(json.loads((RUN_DIR / 'embedding_metadata.json').read_text()))